In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# Import the loader function from your module
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display
import seaborn as sns

pio.renderers.default = 'notebook'

Dask dashboard: /proxy/8787/status


2025-11-18 12:37:03,297 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c277e47be041ff2cfd925b09a29ecef6 initialized by task ('shuffle-transfer-c277e47be041ff2cfd925b09a29ecef6', 1) executed on worker tcp://127.0.0.1:46049
2025-11-18 12:37:23,031 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c277e47be041ff2cfd925b09a29ecef6 deactivated due to stimulus 'task-finished-1763429843.0297287'
2025-11-18 12:37:45,586 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c62a6203d0c02fd9b4469aab50a4633d initialized by task ('shuffle-transfer-c62a6203d0c02fd9b4469aab50a4633d', 0) executed on worker tcp://127.0.0.1:34433
2025-11-18 12:37:49,267 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c62a6203d0c02fd9b4469aab50a4633d deactivated due to stimulus 'task-finished-1763429869.0714173'


In [ ]:
df, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="hourly",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    wind_zero_threshold=40,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.21 sec
Select group: 0.01 sec
--- Starting Dask-Native Process ---


/g/data/ng72/ms5578/ID_HW_BARRA/process_code.py:168: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Starting final Dask compute...


2025-11-18 12:42:37,742 - distributed.worker - ERROR - failed during get data with tcp://127.0.0.1:45819 -> tcp://127.0.0.1:39285
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.08/lib/python3.11/site-packages/tornado/iostream.py", line 861, in _read_to_buffer
    bytes_read = self.read_from_fd(buf)
                 ^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.08/lib/python3.11/site-packages/tornado/iostream.py", line 1113, in read_from_fd
    return self.socket.recv_into(buf, len(buf))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TimeoutError: [Errno 110] Connection timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.08/lib/python3.11/site-packages/distributed/worker.py", line 1795, in get_data
    response = await comm.read(deserializers=serializers)
               ^^^^^^^

In [ ]:
boco = df[df['DUID'] == 'BOCORWF1']

In [ ]:
def highLights(df, fig, variable, level, mode, fillcolor, layer):
    """
    Set a specified color as background for given
    levels of a specified variable using a shape.
    
    Keyword arguments:
    ==================
    fig -- plotly figure
    variable -- column name in a pandas dataframe
    level -- int or float
    mode -- set threshold above or below
    fillcolor -- any color type that plotly can handle
    layer -- position of shape in plotly fiugre, like "below"
    
    """
    
    if mode == 'above':
        m = df[variable].gt(level)
    
    if mode == 'below':
        m = df[variable].lt(level)
        
    df1 = df[m].groupby((~m).cumsum())['time'].agg(['first','last'])

    for index, row in df1.iterrows():
        #print(row['first'], row['last'])
        fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0=row['first'],
            y0=0,
            x1=row['last'],
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(100,100,100,0.2)",
            layer=layer
        )
    return(fig)

In [ ]:
def plot_multivars(df, title='Time Series Plot',lines=['TOTALMWh'], highlight = False):
    fig = go.Figure()

    for line in lines:
        fig.add_trace(go.Scatter(
            x=df['time'],
            y=df[line],
            mode='lines',
            name=str(line)
        ))
        
        if highlight == True:
            # Highlight the EHF flag
            fig = highLights(
                df=df,             # Only this group's data
                fig=fig,
                variable='EHF_flag',  # Column to check
                level=0,              # Threshold
                mode='above',         # or 'below'
                fillcolor='rgba(255,0,0,0.1)',  # Semi-transparent red
                layer='below'
            )

    fig.update_layout(
        title=title,
        xaxis_title='Time',
        yaxis_title='Total MWh',
        template='plotly_white',
        legend_title='Legend'
    )

    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=1,
                         label="1d",
                         step="day",
                         stepmode="backward"),
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )

    return fig

In [ ]:
# plot_multivars(boco,lines=['TOTALMWh','EHF_val'], title='', highlight=True)

In [ ]:
def hist_hw_separate(df):
    df = df.copy()
    df['Hour'] = df['time'].dt.hour
    df['TimeLabel'] = df['Hour'].apply(lambda h: f"{h:02d}:00")

    # Aggregate for EHF_flag = 0
    df0 = df[df['EHF_flag'] == 0].groupby(['Hour', 'TimeLabel'])['TOTALMWh'].sum().reset_index()
    # Aggregate for EHF_flag = 1
    df1 = df[df['EHF_flag'] == 1].groupby(['Hour', 'TimeLabel'])['TOTALMWh'].sum().reset_index()
    # Aggregate total
    dftot = df.groupby(['Hour', 'TimeLabel'])['TOTALMWh'].sum().reset_index()

    # Create subplot with 3 rows, shared x-axis
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=("EHF_flag = 0", "EHF_flag = 1", "Total")
    )

    fig.add_trace(
        go.Bar(x=df0['TimeLabel'], y=df0['TOTALMWh'], name='EHF_flag=0'),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(x=df1['TimeLabel'], y=df1['TOTALMWh'], name='EHF_flag=1'),
        row=2, col=1
    )

    fig.add_trace(
        go.Bar(x=dftot['TimeLabel'], y=dftot['TOTALMWh'], name='Total'),
        row=3, col=1
    )

    fig.update_layout(
        height=900,
        title_text="Boco Rock Sum of Generation by Hour of Day",
        showlegend=False,
        xaxis3_tickangle=-45,
        xaxis2_tickangle=-45,
        xaxis_tickangle=-45,
    )

    fig.update_xaxes(title_text="Hour of Day", row=3, col=1)
    fig.update_yaxes(title_text="Total MWh")

    fig.show()
# hist_hw_separate(boco)

In [ ]:
def line_mean_totalmwh(df):
    df = df.copy()
    df['Hour'] = df['time'].dt.hour
    df['TimeLabel'] = df['Hour'].apply(lambda h: f"{h:02d}:00")

    # Calculate mean TOTALMWh per hour for EHF_flag=0
    mean_0 = df[df['EHF_flag'] == 0].groupby(['Hour', 'TimeLabel'])['TOTALMWh'].mean().reset_index()
    mean_0['Scenario'] = 'EHF_flag = 0'

    # Calculate mean TOTALMWh per hour for EHF_flag=1
    mean_1 = df[df['EHF_flag'] == 1].groupby(['Hour', 'TimeLabel'])['TOTALMWh'].mean().reset_index()
    mean_1['Scenario'] = 'EHF_flag = 1'

    # Calculate mean TOTALMWh per hour for all data
    mean_all = df.groupby(['Hour', 'TimeLabel'])['TOTALMWh'].mean().reset_index()
    mean_all['Scenario'] = 'All'

    # Combine all scenarios
    combined = pd.concat([mean_0, mean_1, mean_all], ignore_index=True)

    fig = px.line(
        combined,
        x='TimeLabel',
        y='TOTALMWh',
        color='Scenario',
        title='Mean TOTALMWh by Hour of Day',
        labels={'TimeLabel': 'Hour of Day', 'TOTALMWh': 'Mean TOTALMWh'}
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

In [ ]:
nsw = df[df['region'] == 'NSW1']

In [ ]:
def plot_loc(df, info, cmap='Set1'):
    unit_summ = df.reset_index().groupby(['DUID','EHF_flag'])['TOTALMWh'].agg(['mean','median','min','max','std','var']).reset_index()
    unit_summ = unit_summ.merge(info, on='DUID',how='left')
    unit_summ['size'] = unit_summ['mean'].abs()
    unit_summ['size'] = unit_summ['size'].replace(0, 0.001)  # replace zeros with small positive value
    unit_summ['size'] = unit_summ['size']*100  # replace zeros with small positive value
    
    in_heatwave = unit_summ[unit_summ['EHF_flag'] == True]
    no_heatwave = unit_summ[unit_summ['EHF_flag'] == False]
    
    # Create traces using px.scatter_map
    fig = px.scatter_map(
        in_heatwave,
        lat='lat_jittered',
        lon='lon_jittered',
        hover_name='DUID',
        hover_data={
            'lat': False,
            'lon': False,
            'lat_jittered': False,
            'lon_jittered': False,
            'fuel_source_primary': True,
            'size':False
        },
        color='mean',
        zoom=5,
        map_style='carto-darkmatter'
    )
    
    fig.update_layout(
        legend=dict(
            title="Fuel Type",
        ),
        title_text="Locations of Generation Units"
    )
    
    fig.show()


In [ ]:
def line_mean_by_unit(df):
    df = df.copy()
    df['Hour'] = df['time'].dt.hour
    df['TimeLabel'] = df['Hour'].apply(lambda h: f"{h:02d}:00")

    duid_list = df['DUID'].unique()
    fig = go.Figure()

    # Store visibility and button config
    visibility = []
    buttons = []

    for i, duid in enumerate(duid_list):
        sub_df = df[df['DUID'] == duid]

        # Calculate mean for each scenario
        mean_0 = sub_df[sub_df['EHF_flag'] == 0].groupby(['Hour', 'TimeLabel'])['TOTALMWh'].mean().reset_index()
        mean_0['Scenario'] = 'EHF_flag = 0'

        mean_1 = sub_df[sub_df['EHF_flag'] == 1].groupby(['Hour', 'TimeLabel'])['TOTALMWh'].mean().reset_index()
        mean_1['Scenario'] = 'EHF_flag = 1'

        mean_all = sub_df.groupby(['Hour', 'TimeLabel'])['TOTALMWh'].mean().reset_index()
        mean_all['Scenario'] = 'All'

        combined = pd.concat([mean_0, mean_1, mean_all], ignore_index=True)

        for scenario in ['EHF_flag = 0', 'EHF_flag = 1', 'All']:
            plot_df = combined[combined['Scenario'] == scenario]
            visible = (i == 0)  # Only show first DUID initially
            fig.add_trace(go.Scatter(
                x=plot_df['TimeLabel'],
                y=plot_df['TOTALMWh'],
                name=f'{scenario} ({duid})',
                visible=visible,
                legendgroup=scenario,
                showlegend=(i == 0)  # Show legend only for the first DUID to avoid clutter
            ))
            visibility.append(visible)

        # Button for this DUID
        vis = [False] * len(duid_list) * 3
        start = i * 3
        vis[start:start+3] = [True, True, True]
        buttons.append(dict(
            label=duid,
            method='update',
            args=[{'visible': vis},
                   {'title': {'text': f'Mean TOTALMWh by Hour of Day for DUID: {duid}'}}]
        ))

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=1.1,
            xanchor='center',
            y=1.15,
            yanchor='top'
        )],
        title=f'Mean TOTALMWh by Hour of Day for DUID: {duid_list[0]}',
        xaxis_title='Hour of Day',
        yaxis_title='Mean TOTALMWh',
        xaxis_tickangle=-45
    )

    fig.show()

In [ ]:
def iqr_ribbon_plot_totalmwh_by_ehf_flag(df, duid):
    df = df[df['DUID'] == duid].copy()
    df['Hour'] = df['time'].dt.hour
    df['TimeLabel'] = df['Hour'].apply(lambda h: f"{h:02d}:00")

    # Prepare figure
    fig = go.Figure()

    # Loop over EHF_flag groups
    for flag_value in sorted(df['EHF_flag'].dropna().unique()):
        group = df[df['EHF_flag'] == flag_value]

        summary = group.groupby(['Hour', 'TimeLabel'])['TOTALMWh'].agg(
            Q1=lambda x: x.quantile(0.25),
            Q3=lambda x: x.quantile(0.75),
            Mean='mean'
        ).reset_index().sort_values('Hour')

        # Q3 line (invisible, top of fill)
        fig.add_trace(go.Scatter(
            x=summary['TimeLabel'], y=summary['Q3'],
            line=dict(width=0),
            showlegend=False,
            name=f'EHF_flag = {flag_value} Q3'
        ))

        # Q1 fill
        fig.add_trace(go.Scatter(
            x=summary['TimeLabel'], y=summary['Q1'],
            fill='tonexty',
            fillcolor=f'rgba({50 + 100 * flag_value}, 100, 255, 0.2)',
            line=dict(width=0),
            name=f'IQR (Q1–Q3), EHF_flag = {flag_value}'
        ))

        # Mean line
        fig.add_trace(go.Scatter(
            x=summary['TimeLabel'], y=summary['Mean'],
            line=dict(width=2),
            name=f'Mean, EHF_flag = {flag_value}'
        ))

    fig.update_layout(
        title=f'IQR + Mean TOTALMWh by Hour for DUID: {duid}',
        xaxis_title='Hour of Day',
        yaxis_title='TOTALMWh',
        xaxis_tickangle=-45,
        template='plotly_white'
    )

    fig.show()


In [ ]:
def iqr_ribbon_plot_totalmwh_dropdown(df):
    df = df.copy()
    df['Hour'] = df['time'].dt.hour
    df['TimeLabel'] = df['Hour'].apply(lambda h: f"{h:02d}:00")

    duids = sorted(df['DUID'].unique())
    fig = go.Figure()
    trace_index = 0
    duid_trace_map = {}  # Keeps track of trace indices for each DUID

    for duid in duids:
        duid_df = df[df['DUID'] == duid]
        duid_traces = []

        for flag_value in sorted(duid_df['EHF_flag'].dropna().unique()):
            group = duid_df[duid_df['EHF_flag'] == flag_value]
            summary = group.groupby(['Hour', 'TimeLabel'])['TOTALMWh'].agg(
                Q1=lambda x: x.quantile(0.25),
                Q3=lambda x: x.quantile(0.75),
                Mean='mean'
            ).reset_index().sort_values('Hour')

            # Q3 (invisible)
            fig.add_trace(go.Scatter(
                x=summary['TimeLabel'], y=summary['Q3'],
                line=dict(width=0),
                showlegend=False,
                visible=(duid == duids[0])  # Only show first DUID initially
            ))
            duid_traces.append(trace_index)
            trace_index += 1

            # Q1 fill
            fig.add_trace(go.Scatter(
                x=summary['TimeLabel'], y=summary['Q1'],
                fill='tonexty',
                fillcolor=f'rgba({50 + 100 * flag_value}, 100, 255, 0.2)',
                line=dict(width=0),
                name=f'IQR (Q1–Q3), EHF_flag = {flag_value}',
                visible=(duid == duids[0])
            ))
            duid_traces.append(trace_index)
            trace_index += 1

            # Mean line
            fig.add_trace(go.Scatter(
                x=summary['TimeLabel'], y=summary['Mean'],
                line=dict(width=2),
                name=f'Mean, EHF_flag = {flag_value}',
                visible=(duid == duids[0])
            ))
            duid_traces.append(trace_index)
            trace_index += 1

        duid_trace_map[duid] = duid_traces

    # Create dropdown buttons
    buttons = []
    total_traces = trace_index

    for duid in duids:
        visibility = [False] * total_traces
        for idx in duid_trace_map[duid]:
            visibility[idx] = True

        buttons.append(dict(
            label=duid,
            method='update',
            args=[
                {'visible': visibility},
                {'title.text': f'IQR + Mean TOTALMWh by Hour for DUID: {duid}'}
                
            ]
        ))

    fig.update_layout(
        updatemenus=[dict(
            buttons=buttons,
            direction='down',
            showactive=True,
            x=1.1,
            xanchor='center',
            y=1.15,
            yanchor='top'
        )],
        title=f'IQR + Mean TOTALMWh by Hour for DUID: {duids[0]}',
        xaxis_title='Hour of Day',
        yaxis_title='TOTALMWh',
        xaxis_tickangle=-45,
        template='plotly_white'
    )

    fig.show()

In [ ]:
iqr_ribbon_plot_totalmwh_dropdown(df)

In [ ]:
def plot_loc_animated_by_hour_filter(df, info, ehf_filter=None, cmap='Turbo'):
    """
    Parameters:
        df: DataFrame with DUID, time, TOTALMWh, EHF_flag
        info: DataFrame with DUID, lat_jittered, lon_jittered, fuel_source_primary
        ehf_filter: 'Heatwave', 'Non-Heatwave', or None (default = all)
        cmap: plotly color scale
    """
    # Copy and prep
    df = df.copy()
    df['Hour'] = df['time'].dt.hour
    df['EHF_flag'] = df['EHF_flag'].map({1: 'Heatwave', 0: 'Non-Heatwave'})

    # Apply heatwave filter if given
    if ehf_filter in ['Heatwave', 'Non-Heatwave']:
        df = df[df['EHF_flag'] == ehf_filter]

    # Aggregate mean generation per hour per DUID
    agg = df.groupby(['DUID', 'EHF_flag', 'Hour'])['TOTALMWh'].mean().reset_index()
    agg.rename(columns={'TOTALMWh': 'mean_gen'}, inplace=True)

    # Merge location info
    agg = agg.merge(info, on='DUID', how='left')

    # Size by mean generation
    agg['size'] = agg['mean_gen'].abs().replace(0, 0.001) * 100
    agg = agg.dropna(subset=['size'])

    # Plot
    fig = px.scatter_map(
        agg,
        lat='lat_jittered',
        lon='lon_jittered',
        size='size',
        color='mean_gen',
        hover_name='DUID',
        hover_data={
            'fuel_source_primary': True,
            'mean_gen': True,
            'EHF_flag': True,
            'Hour': True,
            'lat_jittered': False,
            'lon_jittered': False
        },
        animation_frame='Hour',
        animation_group='DUID',
        zoom=5,
        map_style='carto-positron',
        color_continuous_scale=cmap,
        title=f"Hourly Animated Map of Generator Output ({ehf_filter or 'All Conditions'})"
    )

    fig.update_layout(
        title={'text': f"Hourly Animated Map of Generator Output ({ehf_filter or 'All Conditions'})"},
        margin=dict(l=10, r=10, t=50, b=10)
    )

    fig.show()

In [ ]:
# plot_loc_animated_by_hour_filter(df, info, ehf_filter='Heatwave')

In [ ]:
# plot_loc_animated_by_hour_filter(df, info, ehf_filter='Non-Heatwave')

In [ ]:
def plot_loc_animated_hourly_diff_heatwave_non(df, info, cmap='RdYlGn'):
    """
    Plots the difference in mean generation (heatwave - non-heatwave)
    by hour and DUID on a map with animation. Color scale centered at 0.
    
    Parameters:
        df: DataFrame with ['DUID', 'time', 'TOTALMWh', 'EHF_flag']
        info: DataFrame with location and fuel info
        cmap: Plotly color scale (e.g. 'RdBu', 'Viridis')
    """
    df = df.copy()
    df['Hour'] = df['time'].dt.hour

    # Compute mean generation by DUID, Hour, and EHF_flag
    grouped = df.groupby(['DUID', 'EHF_flag', 'Hour'])['TOTALMWh'].mean().reset_index()
    grouped = grouped.pivot(index=['DUID', 'Hour'], columns='EHF_flag', values='TOTALMWh').reset_index()
    grouped.columns.name = None
    grouped = grouped.rename(columns={0: 'NonHeatwave', 1: 'Heatwave'})

    # Calculate difference: Heatwave - Non-Heatwave
    grouped['Diff'] = grouped['Heatwave'] - grouped['NonHeatwave']

    # Merge location info
    result = grouped.merge(info, on='DUID', how='left')

    # Drop rows with missing difference
    result = result.dropna(subset=['Diff'])

    # Size by magnitude of change
    result['size'] = result['Diff'].abs().replace(0, 0.001) * 100 + 30

    # Center color scale around 0
    max_abs = result['Diff'].abs().max()

    # Plot
    fig = px.scatter_map(
        result,
        lat='lat_jittered',
        lon='lon_jittered',
        size='size',
        color='Diff',
        color_continuous_scale=cmap,
        range_color=[-max_abs, max_abs],
        hover_name='DUID',
        hover_data={
            'Diff': True,
            'Heatwave': True,
            'NonHeatwave': True,
            'fuel_source_primary': True,
            'Hour': True,
            'lat_jittered': False,
            'lon_jittered': False
        },
        animation_frame='Hour',
        animation_group='DUID',
        zoom=5,
        map_style='carto-darkmatter',
        title='Hourly Difference in Mean Output: Heatwave – Non-Heatwave'
    )

    fig.update_layout(
        title={'text': 'Hourly Difference in Mean Output: Heatwave – Non-Heatwave'},
        coloraxis_colorbar=dict(title='Δ MWh'),
        margin=dict(l=10, r=10, t=50, b=10)
    )

    fig.show()

In [ ]:
# plot_loc_animated_hourly_diff_heatwave_non(df, info)

In [ ]:
def plot_loc_animated_hourly_diff_discrete_coloring(df, info, threshold=10):
    import pandas as pd
    import plotly.express as px

    df = df.copy()
    df['Hour'] = df['time'].dt.hour

    # Compute mean generation by DUID, EHF_flag, and Hour
    grouped = df.groupby(['DUID', 'EHF_flag', 'Hour'])['TOTALMWh'].mean().reset_index()
    pivoted = grouped.pivot(index=['DUID', 'Hour'], columns='EHF_flag', values='TOTALMWh').reset_index()
    pivoted.columns.name = None
    pivoted = pivoted.rename(columns={0: 'NonHeatwave', 1: 'Heatwave'})

    # Compute difference
    pivoted['Diff'] = pivoted['Heatwave'] - pivoted['NonHeatwave']

    # Merge with location info
    result = pivoted.merge(info, on='DUID', how='left').dropna(subset=['Diff'])
    result['size'] = result['Diff'].abs().replace(0, 0.001) * 100 + 30

    # Category assignment
    def classify(diff):
        if diff > threshold:
            return f'Increase > {threshold}'
        elif diff < -threshold:
            return f'Decrease < -{threshold}'
        else:
            return f'Change between ±{threshold}'

    result['Category'] = result['Diff'].apply(classify)

    # Add dummy rows for padding
    hours = sorted(result['Hour'].unique())
    categories = [f'Increase > {threshold}', f'Decrease < -{threshold}', f'Change between ±{threshold}']
    dummy_data = []

    for hour in hours:
        existing = result[result['Hour'] == hour]['Category'].unique()
        missing = set(categories) - set(existing)
        for cat in missing:
            dummy_data.append({
                'Hour': hour,
                'Category': cat,
                'lat_jittered': None,
                'lon_jittered': None,
                'size': 0,
                'Diff': 0,
                'DUID': f'dummy_{cat}_{hour}'
            })

    dummy_df = pd.DataFrame(dummy_data)
    result = pd.concat([result, dummy_df], ignore_index=True)

    # Color map
    color_map = {
        f'Increase > {threshold}': 'green',
        f'Decrease < -{threshold}': 'red',
        f'Change between ±{threshold}': 'lightgray'
    }

    # Plot
    fig = px.scatter_map(
        result,
        lat='lat_jittered',
        lon='lon_jittered',
        size='size',
        color='Category',
        color_discrete_map=color_map,
        hover_name='DUID',
        hover_data={
            'Diff': True,
            'Heatwave': True,
            'NonHeatwave': True,
            'fuel_source_primary': True,
            'Hour': True,
            'lat_jittered': False,
            'lon_jittered': False
        },
        animation_frame='Hour',
        animation_group='DUID',
        zoom=5,
        map_style='carto-darkmatter',
        title=f'Hourly Difference in Mean Output (Threshold: ±{threshold} MWh)'
    )

    fig.update_layout(
        title={'text': f'Hourly Difference in Mean Output (Threshold: ±{threshold})'},
        legend_title_text='Change Category',
        margin=dict(l=10, r=10, t=50, b=10)
    )

    fig.show()

In [ ]:
# plot_loc_animated_hourly_diff_discrete_coloring(df, info, 10)

In [ ]:
def plot_hourly_diff_line_by_duid(df):
    """
    Plots a line chart of the hourly difference in mean TOTALMWh (Heatwave - Non-Heatwave)
    for each DUID.
    
    Parameters:
        df: DataFrame with columns ['DUID', 'time', 'TOTALMWh', 'EHF_flag']
    """
    df = df.copy()
    df['Hour'] = df['time'].dt.hour
    df['TimeLabel'] = df['Hour'].apply(lambda h: f"{h:02d}:00")

    # Mean generation by DUID, hour, and EHF_flag
    grouped = df.groupby(['DUID', 'EHF_flag', 'Hour', 'TimeLabel'])['TOTALMWh'].median().reset_index()
    pivoted = grouped.pivot(index=['DUID', 'Hour', 'TimeLabel'], columns='EHF_flag', values='TOTALMWh').reset_index()
    pivoted.columns.name = None
    pivoted = pivoted.rename(columns={0: 'NonHeatwave', 1: 'Heatwave'})

    # Calculate difference
    pivoted['Diff'] = pivoted['Heatwave'] - pivoted['NonHeatwave']

    # Drop units where either value is missing
    pivoted = pivoted.dropna(subset=['Diff'])

    # Plot
    fig = px.line(
        pivoted,
        x='TimeLabel',
        y='Diff',
        color='DUID',
        line_group='DUID',
        title='Hourly Difference in Mean Output (Heatwave - Non-Heatwave) by Unit',
        labels={'TimeLabel': 'Hour of Day', 'Diff': 'Δ MWh'},
    )
    
    fig.update_layout(
        xaxis_tickangle=-45,
        yaxis_title='Heatwave – Non-Heatwave (MWh)',
        template='plotly_white'
    )

    fig.show()

In [ ]:
plot_hourly_diff_line_by_duid(nsw)

In [ ]:
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GUNNING1',
            'CRURWF1',
            'WOODLWN1',
            'BOCORWF1',
            'BODWF1']

cluster = df[df['DUID'].isin(cluster)]

In [ ]:
def yearly_cooccurrence(df, timestamp_col="timestamp", unit_col="unit", flag_col="EHF_flag"):
    """
    Compute year-by-year conditional co-occurrence percentages.
    
    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns [timestamp_col, unit_col, flag_col]
    timestamp_col : str
        Column name with datetime-like timestamps
    unit_col : str
        Column name with unit identifiers
    flag_col : str
        Column name with binary 0/1 flag
    
    Returns
    -------
    pd.DataFrame (long format)
        Columns: [year, unit_1, unit_2, pct_of_unit1, co_occurrences]
    """
    
    # Ensure timestamp is datetime
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    df["year"] = df[timestamp_col].dt.year
    
    results = []
    

    # Pivot to wide format for that year
    wide = df.pivot(index=timestamp_col, columns=unit_col, values=flag_col).fillna(0)
    
    # Count of 1s per unit in this year
    unit_totals = (wide == 1).sum()
    
    # Keep only units with at least one 1
    active_units = unit_totals[unit_totals > 0].index
    wide = wide[active_units]

    
    # Co-occurrence counts
    co_occurrence = pd.DataFrame(0, index=active_units, columns=active_units)
    for u1 in active_units:
        for u2 in active_units:
            co_occurrence.loc[u1, u2] = ((wide[u1] == 1) & (wide[u2] == 1)).sum()
    
    # Conditional percentages (row = unit_1)
    co_occurrence_pct = co_occurrence.div(unit_totals[active_units], axis=0) * 100
    
    # Tidy format
    co_occurrence_df = (
        co_occurrence_pct
        .reset_index(names="unit_1")
        .melt(id_vars="unit_1", var_name="unit_2", value_name="pct_of_unit1")
    )
    
    # Add raw counts too
    co_occurrence_counts = (
        co_occurrence
        .reset_index(names="unit_1")
        .melt(id_vars="unit_1", var_name="unit_2", value_name="co_occurrences")
    )
    
    # Merge percentages with counts
    merged = pd.merge(
        co_occurrence_df,
        co_occurrence_counts,
        on=["unit_1", "unit_2"]
    )
    
    results.append(merged)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        co_occurrence_pct,
        annot=True, fmt=".1f", cmap="YlGnBu", 
        cbar_kws={'label':'% of unit1=1 overlapped'}
    )
    plt.title("Co-occurrence of Heatwave Days")
    plt.ylabel("Reference Unit")
    plt.xlabel("Response Unit")
    plt.savefig("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/wind_chapter/cluster_cooccurence.png")
    plt.show()
    
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

result = yearly_cooccurrence(cluster, timestamp_col="time", unit_col="DUID", flag_col="EHF_flag")
result['co_occur_days'] = result['co_occurrences']/24

result[result['unit_1']=='BOCORWF1'].sort_values(['unit_2'])

In [ ]:
days = cluster.pivot_table(index=['time'], columns='DUID', values='EHF_flag', aggfunc='sum')
clust_hw_days = days[((days['GULLRWF1']==1) | (days['CROOKWF2']==1)| (days['GUNNING1']==1)) & (days['BOCORWF1']==1)]

clust_hw_days = clust_hw_days.groupby(clust_hw_days.index.date).first().reset_index(names='date')
clust_hw_days.to_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv',index=False)
clust_hw_days

In [ ]:
days2 = cluster.pivot_table(index=['time'], columns='DUID', values='EHF_val', aggfunc='sum')

non_hw_days = days[(((days2['GULLRWF1']<=0) | (days2['CROOKWF2']<=0)| (days2['GUNNING1']<=0)) & (days2['BOCORWF1']<=0))]
non_hw_days = non_hw_days.groupby(non_hw_days.index.date).first().reset_index(names='date')
non_hw_days.to_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/no_hw_alpine_cluster.csv',index=False)
non_hw_days

In [ ]:
17208/24